In [1]:
!pip install mediapipe

In [2]:
pip install pyserial


Note: you may need to restart the kernel to use updated packages.


In [1]:
import serial 
import time 

ser = serial.Serial('/dev/ttyUSB0',baudrate=115200,bytesize =8, parity ='N', stopbits =1)
hex = '3A0100020003000400'

data_send = bytes.fromhex(hex)
ser.write(data_send)

ser.close()

In [2]:
def serial_send(hex_value):
    ser = serial.Serial('/dev/ttyUSB0', baudrate=115200, bytesize=8, parity='N', stopbits=1)
    data_to_send = bytes.fromhex(hex_value)
    ser.write(data_to_send)
    ser.close()
    

In [3]:
import cv2
import mediapipe as mp
from google.protobuf.json_format import MessageToDict

# Initializing the Model
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.75,
    min_tracking_confidence=0.75,
    max_num_hands=2)

# Start capturing video from webcam
cap = cv2.VideoCapture(2)

# Variable to keep track of the current hand detected
current_hand = None

while True:
    # Read video frame by frame
    success, img = cap.read()

    # Flip the image(frame)
    img = cv2.flip(img, 1)

    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(imgRGB)

    # If hands are present
    if results.multi_hand_landmarks:
        # Both Hands are present
        if len(results.multi_handedness) == 2:
            # Display 'Both Hands'
            cv2.putText(img, 'Both Hands', (250, 50),
                        cv2.FONT_HERSHEY_COMPLEX,
                        0.9, (0, 255, 0), 2)
            serial_send('3A01000200')
            current_hand = None

        # If any hand present
        else:
            for i in results.multi_handedness:
 
                handedness_dict = MessageToDict(i)                
                # Return whether it is Right or Left Hand
                label = handedness_dict['classification'][0]['label']

                if label == 'Left':
                    # Display 'Left Hand' on left side of window
                    cv2.putText(img, label + ' Hand',
                                (20, 50),
                                cv2.FONT_HERSHEY_COMPLEX,
                                0.9, (0, 255, 0), 2)
                    if current_hand != 'Left':
                        serial_send('3A01000201')
                        current_hand = 'Left'

                if label == 'Right':
                    # Display 'Right Hand' on right side of window
                    cv2.putText(img, label + ' Hand', (460, 50),
                                cv2.FONT_HERSHEY_COMPLEX,
                                0.9, (0, 255, 0), 2)
                    if current_hand != 'Right':
                        serial_send('3A01010200')
                        current_hand = 'Right'

    cv2.imshow('Image', img)
    if cv2.waitKey(1) & 0xff == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()    

2024-09-13 09:03:37.905787: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-09-13 09:03:37.910111: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-13 09:03:37.970334: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-09-13 09:03:38.034601: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-13 09:03:38.118373: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 